In [74]:
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/dataset.csv")

In [50]:
df.sample(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4025,2984-MIIZL,Male,0,No,No,4,Yes,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,Yes,Bank transfer (automatic),74.80,321.9,Yes
5673,6260-XLACS,Male,0,No,No,4,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.70,117.8,No
2682,9462-MJUAW,Male,0,No,No,4,Yes,Yes,DSL,No,...,No,No,No,No,Month-to-month,No,Mailed check,50.40,206.6,Yes
1142,2460-NGXBJ,Male,1,Yes,Yes,11,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Credit card (automatic),75.20,775.3,No
2055,1216-BGTSP,Male,0,No,No,45,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),108.45,4964.7,No


In [51]:
df = df.drop(columns="customerID" , axis=1)
df["SeniorCitizen"] = df["SeniorCitizen"].astype(np.int16)
df["tenure"] = df["tenure"].astype(np.int16)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df["Churn"] = df["Churn"].map({"No":0,"Yes":1})

In [52]:
df.loc[df["TotalCharges"].isnull(), "TotalCharges"] = 0

In [53]:
for col in df.columns:
    print(col, df[col].nunique())

gender 2
SeniorCitizen 2
Partner 2
Dependents 2
tenure 73
PhoneService 2
MultipleLines 3
InternetService 3
OnlineSecurity 3
OnlineBackup 3
DeviceProtection 3
TechSupport 3
StreamingTV 3
StreamingMovies 3
Contract 3
PaperlessBilling 2
PaymentMethod 4
MonthlyCharges 1585
TotalCharges 6531
Churn 2


In [54]:
services = [
    'PhoneService',
    'MultipleLines',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

df['TotalServices'] = (
    df[services] == 'Yes'
).sum(axis=1)

In [55]:
df['AvgMonthlySpend'] = (
    df['TotalCharges'] / (df['tenure'] + 1)
)

In [56]:
df['HighValueCustomer'] = (
    df['MonthlyCharges'] > 80
).astype(int)

In [57]:
df['FiberPremiumRisk'] = (
    (df['InternetService'] == 'Fiber optic') &
    (df['MonthlyCharges'] > 80)
).astype(int)

In [58]:
df['SeniorNoSupport'] = (
    (df['SeniorCitizen'] == 1) &
    (df['TechSupport'] == 'No')
).astype(int)

In [59]:
df['RiskScore'] = (
    (df['Contract'] == 'Month-to-month').astype(int) +
    (df['InternetService'] == 'Fiber optic').astype(int) +
    (df['TechSupport'] == 'No').astype(int) +
    (df['PaymentMethod'] == 'Electronic check').astype(int)
)

In [60]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'TotalServices',
       'AvgMonthlySpend', 'HighValueCustomer', 'FiberPremiumRisk',
       'SeniorNoSupport', 'RiskScore'],
      dtype='object')

In [61]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [66]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

(5634, 36)
(1409, 36)


In [71]:
X_train.to_csv("X_train.csv" , index=False)

In [72]:
X_test.to_csv("X_test.csv" , index=False)
y_train.to_csv("y_train.csv" , index=False)
y_test.to_csv("y_test.csv" , index=False)